In [1]:
import time
import requests
import pandas as pd

HEADERS = {
    # Importante: UA descriptivo (proyecto académico) para reducir bloqueos
    "User-Agent": "TFG-BloodDonation-Research/1.0 (contact: 202009108@alu.comillas.edu)"
}

def reddit_search(subreddit, query, limit_total=200, sort="new"):
    """
    Devuelve una lista de posts (metadata) buscando en un subreddit.
    Usa el endpoint público /r/{subreddit}/search.json
    """
    posts = []
    after = None
    per_page = 100  # máximo típico

    while len(posts) < limit_total:
        params = {
            "q": query,
            "restrict_sr": 1,
            "sort": sort,
            "t": "all",
            "limit": min(per_page, limit_total - len(posts)),
        }
        if after:
            params["after"] = after

        url = f"https://www.reddit.com/r/{subreddit}/search.json"
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if r.status_code != 200:
            print("Search failed:", r.status_code, r.text[:200])
            break

        data = r.json()
        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for c in children:
            d = c.get("data", {})
            posts.append({
                "subreddit": d.get("subreddit"),
                "id": d.get("id"),
                "title": d.get("title"),
                "selftext": d.get("selftext"),
                "created_utc": d.get("created_utc"),
                "num_comments": d.get("num_comments"),
                "score": d.get("score"),
                "permalink": "https://www.reddit.com" + (d.get("permalink") or ""),
                "url": d.get("url"),
            })

        after = data.get("data", {}).get("after")
        if not after:
            break

        time.sleep(1.2)  # rate limit friendly

    return posts

def reddit_get_comments(post_permalink, limit=500):
    """
    Descarga el post + comentarios usando {permalink}.json
    """
    json_url = post_permalink.rstrip("/") + ".json"
    r = requests.get(json_url, headers=HEADERS, params={"limit": limit}, timeout=30)
    if r.status_code != 200:
        print("Comments failed:", r.status_code, json_url)
        return []

    payload = r.json()
    if not isinstance(payload, list) or len(payload) < 2:
        return []

    comments_listing = payload[1].get("data", {}).get("children", [])

    out = []

    def walk(nodes, depth=0):
        for n in nodes:
            kind = n.get("kind")
            d = n.get("data", {})
            if kind != "t1":  # t1=comment
                continue
            out.append({
                "post_permalink": post_permalink,
                "comment_id": d.get("id"),
                "body": d.get("body"),
                "created_utc": d.get("created_utc"),
                "score": d.get("score"),
                "depth": depth,
            })
            replies = d.get("replies")
            if isinstance(replies, dict):
                children = replies.get("data", {}).get("children", [])
                walk(children, depth + 1)

    walk(comments_listing, 0)
    return out

if __name__ == "__main__":
    # Subreddits útiles para el tema (puedes ampliar)
    subreddits = ["spain", "AskSpain", "madrid", "barcelona", "AskReddit", "AskEurope"]
    keywords = [
        '"donar sangre"',
        '"donación de sangre"',
        '"no dono sangre"',
        '"no puedo donar"',
        '"me desmayo"',
        '"miedo a las agujas"',
        '"me da miedo donar"',
        '"tatuaje y donar sangre"',
        '"no me dejan donar"',
    ]


    all_posts = []
    for sr in subreddits:
        for kw in keywords:
            all_posts.extend(reddit_search(sr, kw, limit_total=50, sort="new"))
            time.sleep(1.2)

    # Deduplicar por permalink
    seen = set()
    uniq_posts = []
    for p in all_posts:
        if p["permalink"] and p["permalink"] not in seen:
            uniq_posts.append(p)
            seen.add(p["permalink"])

    print("Posts únicos:", len(uniq_posts))

    all_comments = []
    for i, p in enumerate(uniq_posts, 1):
        if not p["permalink"]:
            continue
        cs = reddit_get_comments(p["permalink"], limit=500)
        all_comments.extend(cs)
        print(f"[{i}/{len(uniq_posts)}] comentarios:", len(cs))
        time.sleep(1.2)

    df_posts = pd.DataFrame(uniq_posts)
    df_comments = pd.DataFrame(all_comments)

    df_posts.to_csv("reddit_posts.csv", index=False)
    df_comments.to_csv("reddit_comments.csv", index=False)

    print("Guardado: reddit_posts.csv y reddit_comments.csv")


Posts únicos: 13
[1/13] comentarios: 117
[2/13] comentarios: 74
[3/13] comentarios: 11
[4/13] comentarios: 0
[5/13] comentarios: 63
[6/13] comentarios: 5
[7/13] comentarios: 9
[8/13] comentarios: 11
[9/13] comentarios: 8
[10/13] comentarios: 10
[11/13] comentarios: 0
[12/13] comentarios: 0
[13/13] comentarios: 0
Guardado: reddit_posts.csv y reddit_comments.csv


## Limpieza de datos extraídos 
Verificación de que los datos extraídos continen cosas relacionadas con la donación de sangre (previo a la lematización o stopwords)

In [2]:
import pandas as pd
import re

In [3]:
posts = pd.read_csv("reddit_posts.csv")
comments = pd.read_csv("reddit_comments.csv")

print("Posts:", posts.shape)
print("Comentarios:", comments.shape)

Posts: (13, 9)
Comentarios: (308, 6)


In [4]:
df = comments.copy()

# Eliminar nulos
df = df[df["body"].notna()]

# Eliminar duplicados
df = df.drop_duplicates(subset="body")

# Eliminar comentarios muy cortos (ruido)
df = df[df["body"].str.len() > 50]

print("Tras limpieza básica:", df.shape)

Tras limpieza básica: (239, 6)


In [5]:
terms = [
    "don", "sangre", "donar",
    "aguja", "pinchazo",
    "miedo", "mareo", "desmayo",
    "tatu", "piercing",
    "anemia", "medico", "medicación",
    "no puedo", "no me dejan", "no dono",
    "requisito", "impiden", "rechazan"
]

pattern = "|".join(terms)

df = df[df["body"].str.lower().str.contains(pattern)]

print("Tras filtrado semántico:", df.shape)

Tras filtrado semántico: (120, 6)


In [6]:
posts["title_clean"] = posts["title"].str.lower().fillna("")

valid_posts = posts[
    posts["title_clean"].str.contains("don")
]["permalink"]

df = df[df["post_permalink"].isin(valid_posts)]

print("Tras filtrar por posts relevantes:", df.shape)

Tras filtrar por posts relevantes: (89, 6)


In [7]:
def basic_text_cleaning(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)      # eliminar URLs
    text = re.sub(r"\n", " ", text)           # saltos de línea
    text = re.sub(r"[^a-záéíóúüñ\s]", "", text)  # solo letras
    text = re.sub(r"\s+", " ", text)          # espacios múltiples
    return text.strip()

df["clean_text"] = df["body"].apply(basic_text_cleaning)

In [8]:
df_clean = df[[
    "post_permalink",
    "comment_id",
    "clean_text"
]].reset_index(drop=True)

print("Corpus final:", df_clean.shape)

df_clean.head()

Corpus final: (89, 3)


,post_permalink,comment_id,clean_text
0,https://www.reddit.com/r/spain/comments/zmkypt...,j0bu7si,no soy español pero vivo aqui desde hace años ...
1,https://www.reddit.com/r/spain/comments/zmkypt...,j0fswe9,donde acudo yo valencia te dan un zumitouna bo...
2,https://www.reddit.com/r/spain/comments/zmkypt...,j0gcqei,oye a mi me gustan los bollos yo personalmente...
3,https://www.reddit.com/r/spain/comments/zmkypt...,j0egslp,no va un poco contra el espíritu de donar algo...
4,https://www.reddit.com/r/spain/comments/zmkypt...,j0f2nyk,cómo llegas a esa conclusión yo creo que es al...


In [9]:
df_clean.to_csv("reddit_comments_clean.csv", index=False)

In [10]:
df_clean.sample(10)["clean_text"].values

array(['sí pero cerca de casa no hay ningún hospital grande antes doné en un puesto de la cruz roja pero no lo quiero repetir',
       'antemano si la cruz roja cobra pavos por bolsa de sangre que dan pero eso es por cubrir costes basicamnete se les paga los costes de recogida manipulacion y transporte de la sangre entiendes que no es exactamente un kebab que lo puede llevar por ahi un glovo es mas para que la gente no venga por eso de la cruz roja cobra por la sagre la cruz roja hace muchas cosas que son una mierda pero esto no es una de ellas',
       'tengo curiosidad por qué el tiempo de espera entre donaciones en hombres es meses y las mujeres',
       'english posts must be directly related to the subreddits main topic and purpose offtopic content including unrelated discussions personal stories that dont connect to the community theme or content that belongs in other subreddits will be removed before posting ask yourself does this clearly relate to what this community is about s

## StopWords + Lematización
Tokenización

Eliminación de stopwords (español)

Lematización

Limpieza final

Preparar corpus para LDA

In [11]:
!pip install spacy
!python -m spacy download es_core_news_sm

^C


C:\Users\Usuario\anaconda3\python.exe: No module named spacy


In [ ]:
import spacy
import pandas as pd

nlp = spacy.load("es_core_news_sm")

df = pd.read_csv("reddit_comments_clean.csv")

# Stopwords adicionales específicas de Reddit
custom_stopwords = set([
    "gente", "persona", "personas", "creo", "hacer",
    "vez", "año", "años", "día", "dias", "donar",
    "donación", "sangre"  # se pueden quitar si dominan demasiado
])

def lemmatize_text(text):
    doc = nlp(text)
    tokens = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.lemma_ not in custom_stopwords
            and len(token.lemma_) > 2
        ):
            tokens.append(token.lemma_)
    return " ".join(tokens)

df["lemmatized_text"] = df["clean_text"].apply(lemmatize_text)

df[["clean_text", "lemmatized_text"]].head()

In [ ]:
df_final = df[["comment_id", "lemmatized_text"]]
df_final = df_final[df_final["lemmatized_text"].str.len() > 0]

df_final.to_csv("reddit_comments_nlp_ready.csv", index=False)

print("Corpus final NLP:", df_final.shape)

## OTRO SCRAPPING MAS AFINADO EN TEORÍA (PROBAR)

In [ ]:
import time
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "TFG-BloodDonation-Research/1.0 (academic use; contact: 202009108@alu.comillas.edu)"
}

# 1) Subreddits: más “cercanos” a España y menos generalistas
SUBREDDITS = [
    "spain",
    "AskSpain",
    "madrid",
    "barcelona"
]

# 2) Keywords: orientadas a preguntas y barreras (mejora MUCHÍSIMO la relevancia)
KEYWORDS = [
    '"puedo donar sangre"',
    '"no puedo donar sangre"',
    '"no dono sangre"',
    '"me rechazaron al donar"',
    '"no me dejaron donar"',
    '"me mareo al donar"',
    '"me desmayo al donar"',
    '"miedo a donar sangre"',
    '"miedo a las agujas"',
    '"donar sangre tatuaje"',
    '"donar sangre piercing"',
    '"donar sangre requisitos"'
]

# 3) Filtros de relevancia para quedarnos con hilos de “donación de sangre” de verdad
REQUIRED_TERMS = ["don", "sangre"]  # deben aparecer en title+selftext
PROBLEM_TERMS = [
    "no puedo", "no dono", "miedo", "mareo", "desmayo",
    "rechaz", "no me dejan", "requisit", "impiden"
]

# 4) Términos opcionales para filtrar comentarios con más densidad temática
COMMENT_TERMS = [
    "don", "sangre", "donar", "aguja", "pinch",
    "miedo", "mareo", "desmayo", "tatu", "piercing",
    "anemia", "medic", "rechaz", "no puedo", "no me dejan",
    "requisit", "impiden"
]


def reddit_search(subreddit, query, limit_total=80, sort="new"):
    """
    Busca posts en un subreddit usando /search.json (sin API auth).
    """
    posts = []
    after = None
    per_page = 100

    while len(posts) < limit_total:
        params = {
            "q": query,
            "restrict_sr": 1,
            "sort": sort,
            "t": "all",
            "limit": min(per_page, limit_total - len(posts)),
        }
        if after:
            params["after"] = after

        url = f"https://www.reddit.com/r/{subreddit}/search.json"
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if r.status_code != 200:
            print("Search failed:", r.status_code, r.text[:200])
            break

        data = r.json()
        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for c in children:
            d = c.get("data", {})
            posts.append({
                "subreddit": d.get("subreddit"),
                "id": d.get("id"),
                "title": d.get("title") or "",
                "selftext": d.get("selftext") or "",
                "created_utc": d.get("created_utc"),
                "num_comments": d.get("num_comments"),
                "score": d.get("score"),
                "permalink": "https://www.reddit.com" + (d.get("permalink") or ""),
                "url": d.get("url"),
            })

        after = data.get("data", {}).get("after")
        if not after:
            break

        time.sleep(1.2)

    return posts


def post_is_relevant(post):
    """
    Filtra posts:
    - Deben hablar de 'don' y 'sangre' (title + selftext)
    - Mejor si incluyen lenguaje de barrera (PROBLEM_TERMS)
    """
    text = (post["title"] + " " + post["selftext"]).lower()

    # requerido: don + sangre
    if not all(t in text for t in REQUIRED_TERMS):
        return False

    # opcional: preferimos posts con lenguaje de "barrera"
    if any(t in text for t in PROBLEM_TERMS):
        return True

    # si no tiene términos de barrera, aún puede ser válido si está muy centrado
    # (por ejemplo “donación de sangre” informativo).
    return True


def reddit_get_comments(post_permalink, limit=500):
    """
    Descarga comentarios de un post usando {permalink}.json
    """
    json_url = post_permalink.rstrip("/") + ".json"
    r = requests.get(json_url, headers=HEADERS, params={"limit": limit}, timeout=30)

    if r.status_code != 200:
        print("Comments failed:", r.status_code, json_url)
        return []

    payload = r.json()
    if not isinstance(payload, list) or len(payload) < 2:
        return []

    comments_listing = payload[1].get("data", {}).get("children", [])
    out = []

    def walk(nodes, depth=0):
        for n in nodes:
            kind = n.get("kind")
            d = n.get("data", {})
            if kind != "t1":
                continue

            body = d.get("body")
            if not body:
                continue

            out.append({
                "post_permalink": post_permalink,
                "comment_id": d.get("id"),
                "body": body,
                "created_utc": d.get("created_utc"),
                "score": d.get("score"),
                "depth": depth,
            })

            replies = d.get("replies")
            if isinstance(replies, dict):
                children = replies.get("data", {}).get("children", [])
                walk(children, depth + 1)

    walk(comments_listing, 0)
    return out


def comment_is_thematic(text):
    """
    Filtro opcional para reducir ruido desde el inicio.
    """
    t = text.lower()
    return any(term in t for term in COMMENT_TERMS)


# =========================
# EJECUCIÓN
# =========================

all_posts = []
for sr in SUBREDDITS:
    for kw in KEYWORDS:
        found = reddit_search(sr, kw, limit_total=60, sort="new")
        all_posts.extend(found)
        time.sleep(1.2)

# Deduplicar por permalink
seen = set()
uniq_posts = []
for p in all_posts:
    if p["permalink"] and p["permalink"] not in seen:
        if post_is_relevant(p):
            uniq_posts.append(p)
            seen.add(p["permalink"])

print("Posts únicos (relevantes):", len(uniq_posts))

# Bajar comentarios
all_comments = []
for i, p in enumerate(uniq_posts, 1):
    cs = reddit_get_comments(p["permalink"], limit=500)

    # filtro opcional: quedarnos con comentarios temáticos desde el inicio
    cs = [c for c in cs if comment_is_thematic(c["body"])]

    all_comments.extend(cs)
    print(f"[{i}/{len(uniq_posts)}] comentarios (filtrados):", len(cs))
    time.sleep(1.2)

df_posts = pd.DataFrame(uniq_posts)
df_comments = pd.DataFrame(all_comments)

df_posts.to_csv("reddit_posts_v2.csv", index=False)
df_comments.to_csv("reddit_comments_v2.csv", index=False)

print("Guardado: reddit_posts_v2.csv y reddit_comments_v2.csv")
print("Total comentarios v2:", df_comments.shape[0])


## Versión 2

In [1]:
import time
import random
import requests
import pandas as pd
from datetime import datetime

HEADERS = {
    "User-Agent": "TFG-BloodDonation-Research/1.0 (academic use; contact: 202009108@alu.comillas.edu)"
}

SUBREDDITS = ["spain", "AskSpain", "madrid", "barcelona"]

KEYWORDS = [
    '"puedo donar sangre"',
    '"no puedo donar sangre"',
    '"no dono sangre"',
    '"me rechazaron al donar"',
    '"no me dejaron donar"',
    '"me mareo al donar"',
    '"me desmayo al donar"',
    '"miedo a donar sangre"',
    '"miedo a las agujas"',
    '"donar sangre tatuaje"',
    '"donar sangre piercing"',
    '"donar sangre requisitos"'
]

REQUIRED_TERMS = ["don", "sangre"]
PROBLEM_TERMS = [
    "no puedo", "no dono", "miedo", "mareo", "desmayo",
    "rechaz", "no me dejan", "requisit", "impiden"
]

COMMENT_TERMS = [
    "don", "sangre", "donar", "aguja", "pinch",
    "miedo", "mareo", "desmayo", "tatu", "piercing",
    "anemia", "medic", "rechaz", "no puedo", "no me dejan",
    "requisit", "impiden"
]

# Ventana temporal (ajusta si quieres)
START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2026, 1, 1)
START_UTC = int(START_DATE.timestamp())
END_UTC = int(END_DATE.timestamp())

LIMIT_TOTAL_PER_QUERY = 60
SORT = "new"
COMMENTS_LIMIT = 300  # 👈 baja un poco para reducir requests/peso
FILTER_COMMENTS_THEMATIC = True

# Base sleep (con jitter)
BASE_SLEEP = 1.6
JITTER = 0.9


def sleep_jitter(mult=1.0):
    time.sleep((BASE_SLEEP + random.random() * JITTER) * mult)


def request_with_retries(url, headers, params=None, timeout=30, max_retries=6):
    """
    Request robusto:
    - Si 429/503/502: backoff exponencial + jitter
    - Si 403: esperamos más (posible bloqueo temporal)
    """
    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=timeout)
        except requests.RequestException as e:
            print(f"Request error ({attempt}/{max_retries}): {e}")
            sleep_jitter(mult=backoff)
            backoff *= 2
            continue

        if r.status_code == 200:
            return r

        if r.status_code in (429, 502, 503):
            print(f"Rate limit/server issue {r.status_code} ({attempt}/{max_retries}). Backoff {backoff:.1f}x")
            sleep_jitter(mult=backoff)
            backoff *= 2
            continue

        if r.status_code == 403:
            print(f"403 Forbidden ({attempt}/{max_retries}). Cooling down harder...")
            sleep_jitter(mult=backoff * 3)
            backoff *= 2
            continue

        # Otros códigos: devolvemos y que el caller lo gestione
        print(f"HTTP {r.status_code}: {r.text[:150]}")
        return r

    return None


def within_time_window(created_utc: int) -> bool:
    if created_utc is None:
        return False
    return START_UTC <= int(created_utc) < END_UTC


def safe_lower(s):
    return (s or "").lower()


def reddit_search(subreddit, query, limit_total=80, sort="new"):
    posts = []
    after = None
    per_page = 100

    while len(posts) < limit_total:
        params = {
            "q": query,
            "restrict_sr": 1,
            "sort": sort,
            "t": "all",
            "limit": min(per_page, limit_total - len(posts)),
        }
        if after:
            params["after"] = after

        url = f"https://www.reddit.com/r/{subreddit}/search.json"
        r = request_with_retries(url, HEADERS, params=params, timeout=30)
        if r is None:
            print("Search failed: too many retries")
            break
        if r.status_code != 200:
            print("Search failed:", r.status_code, r.text[:150])
            break

        data = r.json()
        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for c in children:
            d = c.get("data", {})
            posts.append({
                "subreddit": d.get("subreddit"),
                "post_id": d.get("id"),
                "title": d.get("title") or "",
                "selftext": d.get("selftext") or "",
                "created_utc": d.get("created_utc"),
                "num_comments": d.get("num_comments"),
                "score": d.get("score"),
                "permalink": "https://www.reddit.com" + (d.get("permalink") or ""),
                "url": d.get("url"),
            })

        after = data.get("data", {}).get("after")
        if not after:
            break

        sleep_jitter()

    return posts


def post_is_relevant(post):
    created_utc = post.get("created_utc")
    if not within_time_window(created_utc):
        return False

    text = safe_lower(post.get("title")) + " " + safe_lower(post.get("selftext"))
    if not all(t in text for t in REQUIRED_TERMS):
        return False

    # Preferimos barreras si existen, pero no exigimos
    return True


def reddit_get_comments(post_permalink, post_id=None, limit=300):
    json_url = post_permalink.rstrip("/") + ".json"
    r = request_with_retries(json_url, HEADERS, params={"limit": limit}, timeout=30)
    if r is None:
        print("Comments failed: too many retries", json_url)
        return []
    if r.status_code != 200:
        print("Comments failed:", r.status_code, json_url)
        return []

    payload = r.json()
    if not isinstance(payload, list) or len(payload) < 2:
        return []

    comments_listing = payload[1].get("data", {}).get("children", [])
    out = []

    def walk(nodes, depth=0):
        for n in nodes:
            if n.get("kind") != "t1":
                continue
            d = n.get("data", {})
            body = d.get("body")
            if not body or body in ("[deleted]", "[removed]"):
                continue

            out.append({
                "post_id": post_id,
                "post_permalink": post_permalink,
                "comment_id": d.get("id"),
                "body": body,
                "created_utc": d.get("created_utc"),
                "score": d.get("score"),
                "depth": depth,
            })

            replies = d.get("replies")
            if isinstance(replies, dict):
                walk(replies.get("data", {}).get("children", []), depth + 1)

    walk(comments_listing, 0)
    return out


def comment_is_thematic(text):
    t = safe_lower(text)
    return any(term in t for term in COMMENT_TERMS)


# =========================
# RUN
# =========================

all_posts = []
for sr in SUBREDDITS:
    for kw in KEYWORDS:
        found = reddit_search(sr, kw, limit_total=LIMIT_TOTAL_PER_QUERY, sort=SORT)
        all_posts.extend(found)
        sleep_jitter()

# Deduplicar + filtrar
seen = set()
uniq_posts = []
for p in all_posts:
    permalink = p.get("permalink")
    if not permalink or permalink in seen:
        continue
    if post_is_relevant(p):
        uniq_posts.append(p)
        seen.add(permalink)

print("Posts únicos (relevantes, ventana temporal):", len(uniq_posts))

all_comments = []
for i, p in enumerate(uniq_posts, 1):
    cs = reddit_get_comments(p["permalink"], post_id=p.get("post_id"), limit=COMMENTS_LIMIT)
    if FILTER_COMMENTS_THEMATIC:
        cs = [c for c in cs if comment_is_thematic(c["body"])]
    all_comments.extend(cs)
    print(f"[{i}/{len(uniq_posts)}] comentarios:", len(cs))
    sleep_jitter()

df_posts = pd.DataFrame(uniq_posts)
df_comments = pd.DataFrame(all_comments)

# Refiltrado temporal “por si acaso”
if not df_posts.empty:
    df_posts = df_posts[(df_posts["created_utc"] >= START_UTC) & (df_posts["created_utc"] < END_UTC)]
if not df_comments.empty:
    df_comments = df_comments[(df_comments["created_utc"] >= START_UTC) & (df_comments["created_utc"] < END_UTC)]

df_posts.to_csv("reddit_posts_v4_robust.csv", index=False)
df_comments.to_csv("reddit_comments_v4_robust.csv", index=False)

print("Guardado: reddit_posts_v4_robust.csv y reddit_comments_v4_robust.csv")
print("Total comentarios:", df_comments.shape[0])

Posts únicos (relevantes, ventana temporal): 1
[1/1] comentarios: 0
Guardado: reddit_posts_v4_robust.csv y reddit_comments_v4_robust.csv
Total comentarios: 0


In [2]:
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "TFG-BloodDonation-Research/1.0 (academic use; contact: 202009108@alu.comillas.edu)"
}

POST_URL = "https://www.reddit.com/r/esConversacion/comments/1lzw9hh/alguna_vez_hab%C3%A9is_donado_sangre_para_ayudar_a_la/"

def get_comments(permalink, limit=500):
    json_url = permalink.rstrip("/") + ".json"
    r = requests.get(
        json_url,
        headers=HEADERS,
        params={"limit": limit, "depth": 10, "raw_json": 1},
        timeout=30
    )
    print("Status:", r.status_code, "URL:", json_url)
    if r.status_code != 200:
        print(r.text[:200])
        return []

    payload = r.json()
    comments_listing = payload[1]["data"]["children"]
    out = []

    def walk(nodes, depth=0):
        for n in nodes:
            kind = n.get("kind")
            d = n.get("data", {})

            # Solo comentarios reales
            if kind != "t1":
                continue

            body = d.get("body")
            if not body or body in ("[deleted]", "[removed]"):
                continue

            out.append({
                "comment_id": d.get("id"),
                "body": body,
                "score": d.get("score"),
                "created_utc": d.get("created_utc"),
                "depth": depth
            })

            replies = d.get("replies")
            if isinstance(replies, dict):
                walk(replies["data"]["children"], depth + 1)

    walk(comments_listing, 0)
    return out

comments = get_comments(POST_URL, limit=500)
print("Comentarios descargados:", len(comments))

df = pd.DataFrame(comments)
df.to_csv("comments_single_post.csv", index=False)
print("Guardado: comments_single_post.csv")

# (Opcional) Ver los 5 primeros para comprobar que hay texto
print(df.head(5)[["comment_id", "depth", "score", "body"]])

Status: 200 URL: https://www.reddit.com/r/esConversacion/comments/1lzw9hh/alguna_vez_hab%C3%A9is_donado_sangre_para_ayudar_a_la.json
Comentarios descargados: 57
Guardado: comments_single_post.csv
  comment_id  depth  score                                               body
0    n350r2r      0      7  Yo soy donante de sangre habitual, ya me toca ...
1    n352d8u      1      2  Bueno, ya haces mucho más que otros que no pue...
2    n36igek      0      5  si done varias veces a familiares y familiares...
3    n3ggttl      1      1  Gracias por tu comentario. Para mí suerte, nun...
4    n39uj3u      0      3  Buenas ! Soy donante de sangre y encima 0- que...


In [3]:
import time
import random
import requests
import pandas as pd
from datetime import datetime

HEADERS = {
    "User-Agent": "TFG-BloodDonation-Research/1.0 (academic use; contact: 202009108@alu.comillas.edu)"
}

# ====== CONFIG ======
# Subreddits donde quieres buscar (todo en minúsculas)
TARGET_SUBREDDITS = {"spain", "askspain", "esconversacion", "madrid", "barcelona", "es"}

# Queries amplias (sin comillas para no perder resultados)
SEARCH_QUERIES = [
    "donar sangre",
    "donación de sangre",
    "donacion de sangre",
    "miedo donar sangre",
    "miedo agujas",
    "no puedo donar sangre",
    "rechazaron donar sangre",
    "requisitos donar sangre",
    "tatuaje donar sangre",
    "piercing donar sangre",
]

# Ventana temporal (ajusta si quieres)
START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2026, 1, 1)
START_UTC = int(START_DATE.timestamp())
END_UTC = int(END_DATE.timestamp())

# Cuántos posts máximo por query (sube si quieres)
POSTS_PER_QUERY = 120
SORT = "new"  # "new" te ayuda a priorizar recientes

# Pausas anti-rate-limit
BASE_SLEEP = 1.6
JITTER = 0.9


def sleep_jitter(mult=1.0):
    time.sleep((BASE_SLEEP + random.random() * JITTER) * mult)


def request_with_retries(url, params=None, max_retries=6):
    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        except requests.RequestException as e:
            print(f"Request error ({attempt}/{max_retries}): {e}")
            sleep_jitter(mult=backoff)
            backoff *= 2
            continue

        if r.status_code == 200:
            return r

        if r.status_code in (429, 502, 503):
            print(f"HTTP {r.status_code} ({attempt}/{max_retries}) → backoff {backoff:.1f}x")
            sleep_jitter(mult=backoff)
            backoff *= 2
            continue

        if r.status_code == 403:
            print(f"HTTP 403 ({attempt}/{max_retries}) → cooldown fuerte")
            sleep_jitter(mult=backoff * 3)
            backoff *= 2
            continue

        print(f"HTTP {r.status_code}: {r.text[:150]}")
        return r

    return None


def within_time_window(created_utc):
    if created_utc is None:
        return False
    created_utc = int(created_utc)
    return START_UTC <= created_utc < END_UTC


def global_reddit_search(query, limit_total=120, sort="new"):
    """
    Busca en Reddit globalmente (mucho más robusto que /r/subreddit/search.json):
    https://www.reddit.com/search.json?q=...
    """
    posts = []
    after = None
    per_page = 100

    while len(posts) < limit_total:
        params = {
            "q": query,
            "sort": sort,
            "t": "all",
            "limit": min(per_page, limit_total - len(posts)),
            "raw_json": 1
        }
        if after:
            params["after"] = after

        url = "https://www.reddit.com/search.json"
        r = request_with_retries(url, params=params)
        if r is None or r.status_code != 200:
            break

        data = r.json()
        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for c in children:
            d = c.get("data", {})
            subreddit = (d.get("subreddit") or "").lower()
            created_utc = d.get("created_utc")

            posts.append({
                "query": query,
                "subreddit": subreddit,
                "post_id": d.get("id"),
                "title": d.get("title") or "",
                "selftext": d.get("selftext") or "",
                "created_utc": created_utc,
                "num_comments": d.get("num_comments"),
                "score": d.get("score"),
                "permalink": "https://www.reddit.com" + (d.get("permalink") or ""),
                "url": d.get("url"),
            })

        after = data.get("data", {}).get("after")
        if not after:
            break

        sleep_jitter()

    return posts


def keep_relevant_posts(posts):
    """
    Filtrado suave para quedarnos con links 'correctos':
    - ventana temporal
    - subreddits objetivo
    - aparece 'don' y 'sangre' en (title + selftext)
    """
    filtered = []
    for p in posts:
        if not within_time_window(p.get("created_utc")):
            continue
        if p.get("subreddit") not in TARGET_SUBREDDITS:
            continue

        text = (p.get("title", "") + " " + p.get("selftext", "")).lower()

        # Requisito mínimo: hablar de donación de sangre (suave para no perder)
        if ("don" in text) and ("sangre" in text):
            # opcional: evitar cosas totalmente irrelevantes
            if p.get("permalink"):
                filtered.append(p)

    # Dedup por permalink
    uniq = {}
    for p in filtered:
        uniq[p["permalink"]] = p

    return list(uniq.values())


# ====== RUN (PASO 1) ======
all_found = []
for q in SEARCH_QUERIES:
    found = global_reddit_search(q, limit_total=POSTS_PER_QUERY, sort=SORT)
    all_found.extend(found)
    sleep_jitter()

relevant_posts = keep_relevant_posts(all_found)
print("Links (posts) relevantes y únicos:", len(relevant_posts))

df_links = pd.DataFrame(relevant_posts)

# Añadimos fecha legible (útil para revisar a ojo)
if not df_links.empty:
    df_links["date_utc"] = df_links["created_utc"].apply(
        lambda x: datetime.utcfromtimestamp(int(x)).strftime("%Y-%m-%d") if pd.notna(x) else None
    )

# Ordena por más recientes (más fácil revisar)
df_links = df_links.sort_values(by="created_utc", ascending=False)

df_links.to_csv("reddit_links_step1.csv", index=False)
print("Guardado: reddit_links_step1.csv")

# Muestra 10 para comprobar
print(df_links[["date_utc", "subreddit", "num_comments", "title", "permalink"]].head(10).to_string(index=False))

Links (posts) relevantes y únicos: 2
Guardado: reddit_links_step1.csv
  date_utc subreddit  num_comments                                                                                                             title                                                                                         permalink
2025-07-14        es           246                                                                   Opiniones sobre donar sangre de forma altruista     https://www.reddit.com/r/es/comments/1lzwwg2/opiniones_sobre_donar_sangre_de_forma_altruista/
2025-07-12    madrid            10 La Comunidad de Madrid recuerda la necesidad de donar sangre en verano para garantizar las necesidades sanitarias https://www.reddit.com/r/Madrid/comments/1lxy4cr/la_comunidad_de_madrid_recuerda_la_necesidad_de/


In [4]:
import time
import random
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "TFG-BloodDonation-Research/1.0 (academic use; contact: 202009108@alu.comillas.edu)"
}

# Queries (amplias). Puedes añadir más.
SEARCH_QUERIES = [
    "donar sangre",
    "donación sangre",
    "donacion sangre",
    "miedo donar sangre",
    "requisitos donar sangre",
    "no puedo donar sangre",
    "rechazaron donar sangre",
    "tatuaje donar sangre",
    "piercing donar sangre",
]

POSTS_PER_QUERY = 300   # sube/baja según quieras
SORT = "new"            # "new" o "relevance"
BASE_SLEEP = 1.5
JITTER = 0.8


def sleep_jitter(mult=1.0):
    time.sleep((BASE_SLEEP + random.random() * JITTER) * mult)


def request_with_retries(url, params=None, max_retries=6):
    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        except requests.RequestException as e:
            print(f"Request error ({attempt}/{max_retries}): {e}")
            sleep_jitter(mult=backoff)
            backoff *= 2
            continue

        if r.status_code == 200:
            return r

        if r.status_code in (429, 502, 503):
            print(f"HTTP {r.status_code} ({attempt}/{max_retries}) → backoff {backoff:.1f}x")
            sleep_jitter(mult=backoff)
            backoff *= 2
            continue

        if r.status_code == 403:
            print(f"HTTP 403 ({attempt}/{max_retries}) → cooldown fuerte")
            sleep_jitter(mult=backoff * 3)
            backoff *= 2
            continue

        print(f"HTTP {r.status_code}: {r.text[:150]}")
        return r

    return None


def global_reddit_search(query, limit_total=300, sort="new"):
    posts = []
    after = None
    per_page = 100

    while len(posts) < limit_total:
        params = {
            "q": query,
            "sort": sort,
            "t": "all",
            "limit": min(per_page, limit_total - len(posts)),
            "raw_json": 1
        }
        if after:
            params["after"] = after

        url = "https://www.reddit.com/search.json"
        r = request_with_retries(url, params=params)
        if r is None or r.status_code != 200:
            break

        data = r.json()
        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for c in children:
            d = c.get("data", {})
            posts.append({
                "query": query,
                "subreddit": (d.get("subreddit") or "").lower(),
                "post_id": d.get("id"),
                "title": d.get("title") or "",
                "created_utc": d.get("created_utc"),
                "num_comments": d.get("num_comments"),
                "score": d.get("score"),
                "permalink": "https://www.reddit.com" + (d.get("permalink") or ""),
                "url": d.get("url"),
            })

        after = data.get("data", {}).get("after")
        if not after:
            break

        sleep_jitter()

    return posts


def title_matches(title: str) -> bool:
    """
    SOLO filtro por título:
    - Debe contener 'sangre'
    - y alguna forma de 'donar/donación/donacion'
    """
    t = (title or "").lower()
    has_sangre = "sangre" in t
    has_donation = ("donar" in t) or ("donación" in t) or ("donacion" in t)
    return has_sangre and has_donation


# ===== RUN =====
all_posts = []
for q in SEARCH_QUERIES:
    all_posts.extend(global_reddit_search(q, limit_total=POSTS_PER_QUERY, sort=SORT))
    sleep_jitter()

# Filtrar SOLO por título (lo que tú quieres)
filtered = [p for p in all_posts if title_matches(p["title"]) and p.get("permalink")]

# Deduplicar por permalink
uniq = {}
for p in filtered:
    uniq[p["permalink"]] = p
links = list(uniq.values())

df_links = pd.DataFrame(links)

# Guardar CSV solo-links
df_links.to_csv("reddit_links_only_title.csv", index=False)

print("Total links encontrados (título match):", len(df_links))
print("Guardado: reddit_links_only_title.csv")
print(df_links[["subreddit", "num_comments", "title", "permalink"]].head(15).to_string(index=False))

Total links encontrados (título match): 219
Guardado: reddit_links_only_title.csv
            subreddit  num_comments                                                                                   title                                                                                                         permalink
lasaventurasdeenrique             0 como que que... que para "no donar sangre" la revista esa de JAL... puros marihuanos... https://www.reddit.com/r/LasAventurasDeEnrique/comments/1qmb0dq/como_que_que_que_para_no_donar_sangre_la_revista/
               malaga             5                                   ¿Hacen falta donaciones de sangre para los afectados?                       https://www.reddit.com/r/Malaga/comments/1qhkad5/hacen_falta_donaciones_de_sangre_para_los/
        latinoamerica             0                                       Donación de Sangre en Astoria, NY: 21 de Febrero!           https://www.reddit.com/r/latinoamerica/comments/1qczstt/donación_de_sa

## API Instagram: SteadyAPI que es free

Prueba1 de instargram API: 1527|UpdPKzEuSxxd0eyEP0DVG4qNLnICbwXkr81BTgoA


In [5]:
import os
import json
import time
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests
import pandas as pd

In [6]:
import requests

AUTH_KEY = "1527|UpdPKzEuSxxd0eyEP0DVG4qNLnICbwXkr81BTgoA"

url = "https://api.steadyapi.com/v1/instagram/users/search"
params = {"search": "cruzrojadonasangre"}  # prueba con algo
headers = {"Authorization": f"Bearer {AUTH_KEY}", "Accept": "application/json"}

r = requests.get(url, headers=headers, params=params, timeout=60)
print("status:", r.status_code)
data = r.json()
data

status: 200


{'meta': {'version': 'v1.0',
  'status': 200,
  'copywrite': 'https://steadyapi.com',
  'search': 'cruzrojadonasangre',
  'total': 0},
 'body': []}

In [8]:
import requests

AUTH_KEY = "1527|UpdPKzEuSxxd0eyEP0DVG4qNLnICbwXkr81BTgoA"
headers = {
    "Authorization": f"Bearer {AUTH_KEY}",
    "Accept": "application/json"
}

r = requests.get(
    "https://api.steadyapi.com/v1/instagram/posts",
    headers=headers,
    params={"username": "cruzrojadonasangre"},
    timeout=60
)

print("status:", r.status_code)
data = r.json()
data

status: 200


{'meta': {'version': 'v1.0',
  'status': 200,
  'copywrite': 'https://steadyapi.com',
  'pagination_token': None,
  'count': 0},
 'body': []}

In [9]:
import requests

AUTH_KEY = "1527|UpdPKzEuSxxd0eyEP0DVG4qNLnICbwXkr81BTgoA"
headers = {
    "Authorization": f"Bearer {AUTH_KEY}",
    "Accept": "application/json"
}

r = requests.get(
    "https://api.steadyapi.com/v1/instagram/search",
    headers=headers,
    params={"search": "donar sangre"},
    timeout=60
)

print("status:", r.status_code)
data = r.json()
data

status: 200


{'meta': {'version': 'v1.0',
  'status': 200,
  'copywrite': 'https://steadyapi.com',
  'search': 'donar sangre',
  'pagination_token': '-'},
 'body': []}

In [10]:
import requests

AUTH_KEY = "1527|UpdPKzEuSxxd0eyEP0DVG4qNLnICbwXkr81BTgoA"  # usa tu key (idealmente una nueva si rotaste)
headers = {"Authorization": f"Bearer {AUTH_KEY}", "Accept": "application/json"}

r = requests.get(
    "https://api.steadyapi.com/v1/instagram/posts",
    headers=headers,
    params={"username": "mrbeast"},
    timeout=60
)

print("status:", r.status_code)
data = r.json()
print("count:", data.get("meta", {}).get("count"))
data

status: 200
count: 12


{'meta': {'version': 'v1.0',
  'status': 200,
  'copywrite': 'https://steadyapi.com',
  'pagination_token': 'MzgwNTY0NjIwMDMyMjQ4NDIwN18yMjc4MTY5NDE1IzIyNzgxNjk0MTU=',
  'count': 12},
 'body': [{'id': '3511510130787749851',
   'shortcode': 'DC7Y9vyyg_b',
   'media_type': 8,
   'product_type': 'carousel_container',
   'taken_at': 1732824649,
   'caption': 'We are both dropping big videos together November 30th 👀',
   'like_count': 19591222,
   'comment_count': 178396,
   'play_count': None,
   'ig_play_count': None,
   'fb_play_count': None,
   'reshare_count': 0,
   'media_url': 'https://scontent-atl3-1.cdninstagram.com/v/t51.29350-15/468618718_1105623314294222_3860806756476331846_n.jpg?stp=dst-jpg_e35_p1080x1080_tt6&efg=eyJ2ZW5jb2RlX3RhZyI6IkNBUk9VU0VMX0lURU0uaW1hZ2VfdXJsZ2VuLjE0NDB4MTgwMC5zZHIuZjI5MzUwLmRlZmF1bHRfaW1hZ2UuYzIifQ&_nc_ht=scontent-atl3-1.cdninstagram.com&_nc_cat=1&_nc_oc=Q6cZ2QGh4aN1ohmCws5ti30fk-5u1eFofinWUxKbKY5GYFS25F0F1Lx4PCNnWVO20fgoZ0s&_nc_ohc=MUbJIC5i7DYQ7kNvwEEUq

In [11]:
import requests

AUTH_KEY = "1527|UpdPKzEuSxxd0eyEP0DVG4qNLnICbwXkr81BTgoA"
headers = {"Authorization": f"Bearer {AUTH_KEY}", "Accept": "application/json"}

r = requests.get(
    "https://api.steadyapi.com/v1/instagram/users/search",
    headers=headers,
    params={"search": "dona"},
    timeout=60
)

print("status:", r.status_code)
data = r.json()
print("count:", data.get("meta", {}).get("count"), "total:", data.get("meta", {}).get("total"))
data["body"][:10]

status: 200
count: None total: 0


[]

In [12]:
import requests

AUTH_KEY = "1527|UpdPKzEuSxxd0eyEP0DVG4qNLnICbwXkr81BTgoA"
headers = {"Authorization": f"Bearer {AUTH_KEY}", "Accept": "application/json"}

candidatos = [
    # Cruz Roja y similares (pueden o no funcionar en SteadyAPI)
    "cruzrojaespana",
    "cruzroja",
    "cruzrojadonasangre",
    "cruzrojamadrid",
    "cruzrojacatalunya",
    "cruzrojavalencia",
    
    # Centros / campañas (algunos existen, otros no; no pasa nada)
    "donasangrevida",
    "gvadonasang",
    "12octubre_dona",
    "donarsangre",            # genérico
    "donaciondesangre",       # genérico
    "donantesdesangre",       # genérico
    
    # Si quieres, mete también alguna cuenta grande de salud (para asegurar “funciona”)
    "sanidadgob",
    "gobiernodeespana",
]

resultados = []

for u in candidatos:
    r = requests.get(
        "https://api.steadyapi.com/v1/instagram/posts",
        headers=headers,
        params={"username": u},
        timeout=60
    )
    data = r.json()
    count = data.get("meta", {}).get("count")
    resultados.append((u, r.status_code, count))
    print(u, "->", r.status_code, "count:", count)

resultados

cruzrojaespana -> 200 count: 0
cruzroja -> 200 count: 0
cruzrojadonasangre -> 200 count: 0
cruzrojamadrid -> 200 count: 12
cruzrojacatalunya -> 200 count: 0
cruzrojavalencia -> 200 count: 12
donasangrevida -> 200 count: 12
gvadonasang -> 200 count: 12
12octubre_dona -> 200 count: 12
donarsangre -> 200 count: 0
donaciondesangre -> 200 count: 0
donantesdesangre -> 200 count: 0
sanidadgob -> 200 count: 12
gobiernodeespana -> 200 count: 0


[('cruzrojaespana', 200, 0),
 ('cruzroja', 200, 0),
 ('cruzrojadonasangre', 200, 0),
 ('cruzrojamadrid', 200, 12),
 ('cruzrojacatalunya', 200, 0),
 ('cruzrojavalencia', 200, 12),
 ('donasangrevida', 200, 12),
 ('gvadonasang', 200, 12),
 ('12octubre_dona', 200, 12),
 ('donarsangre', 200, 0),
 ('donaciondesangre', 200, 0),
 ('donantesdesangre', 200, 0),
 ('sanidadgob', 200, 12),
 ('gobiernodeespana', 200, 0)]